ran once to make the csv files

In [ ]:
import pandas as pd
from pathlib import path

In [ ]:
ROOT = Path("datasets")

SMALL_DIR = Path("datasets-small")
MEDIUM_DIR = Path("datasets-medium")
LARGE_DIR = Path("datasets-large")

SMALL_DIR.mkdir(exist_ok=True)
MEDIUM_DIR.mkdir(exist_ok=True)
LARGE_DIR.mkdir(exist_ok=True)

SKIP_FILES = {
    "holidays_events.csv",
    "items.csv",
    "oil.csv",
    "stores.csv",
    "calendar.csv",
    "sell_prices.csv",
    "sample_submission.csv",
    "M4-info.csv",
    "train.csv",
    "test.csv"
}

for csv_file in ROOT.rglob("*.csv"):

    if csv_file.name in SKIP_FILES:
        print(f"Skipping {csv_file}")
        continue

    try:
        df = pd.read_csv(csv_file)

        n = len(df)

        small_n = max(1, int(n * 0.10))
        medium_n = max(1, int(n * 0.50))
        large_n = n

        small_df = df.iloc[:small_n]
        medium_df = df.iloc[:medium_n]
        large_df = df

        relative_path = csv_file.relative_to(ROOT)

        small_path = SMALL_DIR / relative_path
        medium_path = MEDIUM_DIR / relative_path
        large_path = LARGE_DIR / relative_path

        small_path.parent.mkdir(parents=True, exist_ok=True)
        medium_path.parent.mkdir(parents=True, exist_ok=True)
        large_path.parent.mkdir(parents=True, exist_ok=True)

        small_df.to_csv(small_path, index=False)
        medium_df.to_csv(medium_path, index=False)
        large_df.to_csv(large_path, index=False)

        print(
            f"\n{relative_path}\n"
            f"Original : {n:,}\n"
            f"Small    : {small_n:,} (10%)\n"
            f"Medium   : {medium_n:,} (50%)\n"
            f"Large    : {large_n:,} (100%)"
        )

    except Exception as e:
        print(f"Failed processing {csv_file}")
        print(e)

print("\nAll datasets processed successfully.")

In [ ]:
train = pd.read_csv(
    "datasets/favorita-retail-dataset/train.csv",
    usecols=[
        "date",
        "store_nbr",
        "item_nbr",
        "unit_sales",
        "onpromotion"
    ]
)

test = pd.read_csv(
    "datasets/favorita-retail-dataset/test.csv",
    usecols=[
        "id",
        "date",
        "store_nbr",
        "item_nbr",
        "onpromotion"
    ]
)

# Load transactions
transactions = pd.read_csv(
    "datasets/favorita-retail-dataset/transactions.csv"
)

# Create series id
train["series_id"] = (
    train["store_nbr"].astype(str)
    + "_"
    + train["item_nbr"].astype(str)
)

series_ids = sorted(
    train["series_id"].unique()
)

# Series sizes
small_ids = series_ids[:100]
medium_ids = series_ids[:1000]
large_ids = series_ids[:5000]

# Train splits
favorita_small_train = train[
    train["series_id"].isin(small_ids)
]

favorita_medium_train = train[
    train["series_id"].isin(medium_ids)
]

favorita_large_train = train[
    train["series_id"].isin(large_ids)
]

# Stores represented in each split
small_stores = favorita_small_train["store_nbr"].unique()

medium_stores = favorita_medium_train["store_nbr"].unique()

large_stores = favorita_large_train["store_nbr"].unique()

# Corresponding transaction subsets
favorita_small_transactions = transactions[
    transactions["store_nbr"].isin(small_stores)
]

favorita_medium_transactions = transactions[
    transactions["store_nbr"].isin(medium_stores)
]

favorita_large_transactions = transactions[
    transactions["store_nbr"].isin(large_stores)
]

# Save
favorita_small_train.to_csv(
    "datasets-small/favorita-retail-dataset/train.csv",
    index=False
)

favorita_medium_train.to_csv(
    "datasets-medium/favorita-retail-dataset/train.csv",
    index=False
)

favorita_large_train.to_csv(
    "datasets-large/favorita-retail-dataset/train.csv",
    index=False
)

favorita_small_transactions.to_csv(
    "datasets-small/favorita-retail-dataset/transactions.csv",
    index=False
)

favorita_medium_transactions.to_csv(
    "datasets-medium/favorita-retail-dataset/transactions.csv",
    index=False
)

favorita_large_transactions.to_csv(
    "datasets-large/favorita-retail-dataset/transactions.csv",
    index=False
)